# 03 — Audio Quality Control

Reproduit la section **Technical Validation** du papier (Nsumba et al., 2026) :

| Check du papier | Ce qu'on fait |
|---|---|
| Decodability & integrity | Décode chaque audio, vérifie sample rate / durée / canaux |
| Silence & dropouts | RMS par frame, flag si médiane sous un seuil (micro défaillant) |
| Spectral sanity | Énergie par bande : 0–250 Hz / 250 Hz–2 kHz / 2–8 kHz, outliers > 3 MAD |
| Duplication | Hash MD5 sur le contenu audio |

On travaille sur un échantillon (l'audio est lourd) — la logique est identique sur le dataset complet.

In [ ]:
from datasets import load_dataset, Audio
import numpy as np
import pandas as pd
import librosa
import hashlib
import matplotlib.pyplot as plt

# Token HF chargé depuis .env (gitignoré)
from dotenv import load_dotenv
import os
load_dotenv('../.env')
HF_TOKEN = os.environ['HF_TOKEN']

# Streaming pour ne pas télécharger 3 GB — on prend N_AUDIO échantillons
N_AUDIO = 200
ds = load_dataset('Sunbird/urban-noise-uganda-61k', 'small', split='train',
                  streaming=True, token=HF_TOKEN)
ds = ds.cast_column('audio', Audio(sampling_rate=16000))

In [ ]:
# ---- Check 1+2+3+4 en une passe ----
RMS_SILENCE_THRESHOLD = 1e-4   # médiane RMS sous ce seuil = suspect (micro mort)

def band_energy(y, sr):
    """Énergie dans les 3 bandes du papier : 0-250 Hz, 250-2k, 2k-8k."""
    S = np.abs(np.fft.rfft(y)) ** 2
    freqs = np.fft.rfftfreq(len(y), 1 / sr)
    total = S.sum() + 1e-12
    return (
        S[freqs < 250].sum() / total,
        S[(freqs >= 250) & (freqs < 2000)].sum() / total,
        S[(freqs >= 2000) & (freqs < 8000)].sum() / total,
    )

rows = []
for i, ex in enumerate(ds):
    if i >= N_AUDIO:
        break
    try:
        y = ex['audio']['array']
        sr = ex['audio']['sampling_rate']
        rms_frames = librosa.feature.rms(y=y)[0]
        low, mid, high = band_energy(y, sr)
        rows.append({
            'idx': i,
            'decodable': True,
            'duration_s': len(y) / sr,
            'sr': sr,
            'median_rms': float(np.median(rms_frames)),
            'band_low': low, 'band_mid': mid, 'band_high': high,
            'md5': hashlib.md5(y.tobytes()).hexdigest(),
            'noise_dB': ex.get('noise_measurement'),
            'class': ex.get('class'),
        })
    except Exception as e:
        rows.append({'idx': i, 'decodable': False, 'error': str(e)})

qc = pd.DataFrame(rows)
print(f"{len(qc)} fichiers analysés — {qc['decodable'].sum()} décodables")

In [ ]:
# ---- Flags qualité (mêmes critères que le papier) ----
ok = qc[qc['decodable']].copy()

# Silence / micro mort
ok['flag_silence'] = ok['median_rms'] < RMS_SILENCE_THRESHOLD

# Outliers spectraux : > 3 MAD de la médiane (critère exact du papier)
for band in ['band_low', 'band_mid', 'band_high']:
    med = ok[band].median()
    mad = (ok[band] - med).abs().median()
    ok[f'flag_{band}'] = (ok[band] - med).abs() > 3 * mad

# Doublons exacts par hash
ok['flag_duplicate'] = ok.duplicated(subset='md5', keep='first')

flags = [c for c in ok.columns if c.startswith('flag_')]
ok['any_flag'] = ok[flags].any(axis=1)

print('Résumé QC :')
for f in flags:
    print(f'  {f:18s}: {ok[f].sum():4d} fichiers')
print(f'  {"TOTAL flaggés":18s}: {ok["any_flag"].sum()} / {len(ok)}')

In [ ]:
# Visualisation : distribution RMS et énergie par bande
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(np.log10(ok['median_rms'] + 1e-12), bins=30, color='steelblue')
axes[0].axvline(np.log10(RMS_SILENCE_THRESHOLD), color='red', linestyle='--', label='Seuil silence')
axes[0].set_title('log10(RMS médian)')
axes[0].legend()

ok[['band_low', 'band_mid', 'band_high']].boxplot(ax=axes[1])
axes[1].set_title('Énergie par bande (fraction)')

axes[2].hist(ok['duration_s'], bins=30, color='steelblue')
axes[2].set_title('Durée (s) — papier : >= 10 s')

plt.tight_layout()
plt.savefig('../outputs/maps/audio_qc.png', dpi=150)
plt.show()

# Sauvegarde la liste propre (sans flags) pour la suite
ok[~ok['any_flag']].to_csv('../data/processed/audio_qc_passed.csv', index=False)
print(f"{(~ok['any_flag']).sum()} fichiers valides sauvegardés")

## Note pour les mesures Hanoï

La même cellule QC tournera sur vos enregistrements terrain — il suffira de remplacer
la source par vos fichiers `.wav`/`.ogg` dans `data/raw/hanoi/`.